# SecureFinger — extended experiments
Runtime → Change runtime type → **T4 GPU**. Run the cells top to bottom. Results are saved to Google Drive, so they survive a disconnect.

## 1. Upload the code (`securefinger.zip`) and install

In [ ]:
from google.colab import files
files.upload()  # choose securefinger.zip
!unzip -q -o securefinger.zip
%cd /content/securefinger
!pip -q install tenseal fastapi httpx cryptography opencv-python-headless pyjwt scikit-learn

## 2. Mount Google Drive (results are written there)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
OUT = '/content/drive/MyDrive/securefinger_results_ext'
!mkdir -p "$OUT"

## 3. Get SOCOFing
Use **ONE** of the two options below, then check that `DATA` lists `Altered` and `Real`.

In [ ]:
# Option A: Kaggle API token (Kaggle -> Settings -> API -> Generate New Token)
import os
os.environ['KAGGLE_API_TOKEN'] = 'PASTE_YOUR_TOKEN_HERE'
!pip -q install -U kaggle
!kaggle datasets download -d ruizgara/socofing -p /content
!unzip -q -o /content/socofing.zip -d /content/socofing_raw

In [ ]:
# Option B: the dataset zip is already in your Google Drive (e.g. MyDrive/archive.zip)
!unzip -q -o "/content/drive/MyDrive/archive.zip" -d /content/socofing_raw

In [ ]:
!find /content/socofing_raw -maxdepth 3 -type d | head
DATA = '/content/socofing_raw/SOCOFing'
!ls $DATA

## 4. Quick check (about 3 minutes)
One seed, one epoch, unlinkability and Procrustes skipped. It only checks that everything runs.

In [ ]:
!python experiments/experiments.py --data $DATA --out /content/check --seeds 0 --epochs 1 --skip easyonly link procrustes

## 5. Full run (about 60–75 minutes)
Trains 3 seeds × 2 configurations (15 epochs each), then the unlinkability study and the Procrustes attack. Keep the tab open.

In [ ]:
!python experiments/experiments.py --data $DATA --out "$OUT" --seeds 0 1 2 --epochs 15 2>&1 | tee "$OUT/log.txt"

## 6. Summary to paste back

In [ ]:
import json
r = json.load(open(f'{OUT}/experiments.json'))
for c in ('full', 'easyonly'):
    if c in r:
        print('==', c)
        for k, v in r[c]['summary'].items():
            print(f"  {k:40s} {v['mean']*100:8.4f}% ± {v['std']*100:.4f}")
print(json.dumps({k: r[k] for k in ('unlinkability', 'procrustes') if k in r}, indent=1))

## 7. Download everything

In [ ]:
!cd "$OUT" && zip -q -r /content/results_ext.zip . -x '*.pt'
from google.colab import files
files.download('/content/results_ext.zip')